# Extraction Parity Verification Notebook

This notebook validates that the extracted `run_context.json` matches the evidence in ROBERT's source output files (PREDICT_data.dat and VERIFY_data.dat).

## What is Parity?

**Parity** means every value in `run_context.json` has a corresponding source in ROBERT's dat files. If the extraction is correct, parity checks will pass. If parity fails, there's a bug in the extraction logic.

## Use Cases

1. **Interactive Audit**: Run cell-by-cell to inspect extraction logic and debug issues.
2. **Programmatic**: Call the `check_parity()` function from Python to verify a run before chat is enabled in the UI.
3. **CI/Test**: Ensure extraction stays in sync with ROBERT outputs after code changes.

## Parity Status Meanings

- **"pass"**: All critical evidence fields extracted and match source files. Safe to use for diagnosis.
- **"incomplete"**: Run lacks VERIFY/CURATE/GENERATE files, but available evidence is correct. Safe but with limited fields.
- **"fail"**: Extracted values do not match source. Extraction bug. Do not use for diagnosis.

## Evidence Mapping (DAT Files -> UI Ground Truth -> Report Sections)

This notebook treats DAT files as deterministic evidence and `run_context.json` as UI ground truth.

- **PREDICT_data.dat** -> `run_context.predict.*` -> model performance sections (CV/Test metrics, descriptor counts).
- **VERIFY_data.dat** -> `run_context.verify.*` -> ROBERT verification sections (y_mean/y_shuffle/onehot verdicts, sorted CV arrays).
- **CURATE_data.dat** -> `run_context.curate.*` (when available) -> data curation/provenance context.
- **GENERATE_data.dat** -> `run_context.generate.*` (when available) -> model generation/provenance context.

The UI parity safeguard checks that the values loaded by the UI still match the DAT evidence before chat is enabled.

## Cell 2: Import Required Libraries

Import Python tools for file I/O, regex parsing, and JSON handling.

In [ ]:
import json
import re
import ast
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

print("✓ Libraries imported")

## Cell 3: Define Parity Result Data Structure

Define the result structure that will be returned by the parity check. This makes the output auditable and machine-readable.

In [ ]:
@dataclass
class ParityResult:
    """Result of a parity check.
    
    Attributes:
        run_id: Identifier for the ROBERT run
        pred_type: Prediction type ("reg" or "clas")
        parity_status: One of "pass", "fail", "incomplete"
        gaps: List of fields that were expected but not extracted
        failures: List of fields where extracted value != source value
        messages: Human-readable status messages
    """
    run_id: str
    pred_type: Optional[str]
    parity_status: str  # "pass", "fail", or "incomplete"
    gaps: List[str]
    failures: List[str]
    messages: List[str]
    
    def to_dict(self) -> dict:
        """Convert to dict for JSON serialization."""
        return {
            "run_id": self.run_id,
            "pred_type": self.pred_type,
            "parity_status": self.parity_status,
            "gaps": self.gaps,
            "failures": self.failures,
            "messages": self.messages,
        }

print("✓ ParityResult dataclass defined")

## Cell 4: Helper Functions for Parsing

Define safe parsing helpers used when extracting values from ROBERT dat files.

In [ ]:
def safe_float(value_str, label=""):
    """Convert string to float, return None if it fails."""
    if value_str is None:
        return None
    try:
        return float(value_str.strip())
    except (ValueError, AttributeError):
        return None

def safe_int(value_str, label=""):
    """Convert string to int, return None if it fails."""
    if value_str is None:
        return None
    try:
        return int(value_str.strip())
    except (ValueError, AttributeError):
        return None

def split_into_blocks(lines, no_pfi_marker, pfi_marker):
    """Split dat file lines into No PFI and PFI blocks."""
    blocks = {"no_pfi": [], "pfi": []}
    if lines is None:
        return blocks

    current = None
    for line in lines:
        if no_pfi_marker in line:
            current = "no_pfi"
        elif pfi_marker in line:
            current = "pfi"
        if current is not None:
            blocks[current].append(line)

    return blocks

print("✓ Helper functions defined")

## Cell 5: Extract PREDICT Evidence from Source Files

Parse PREDICT_data.dat to extract the expected values that should be in run_context.

In [ ]:
def extract_predict_parity(predict_dat_path: Path) -> dict:
    """Extract PREDICT metrics from dat file for parity comparison.
    
    Returns dict with keys: no_pfi, pfi
    Each contains: n_train, n_test, n_descriptors, r2_cv, r2_test, rmse_cv, rmse_test,
                   mcc_cv, mcc_test (for classification), etc.
    """
    with open(predict_dat_path) as f:
        lines = f.readlines()
    
    NO_PFI_MARKER = "Starting model with all variables (No PFI)"
    PFI_MARKER = "Starting model with PFI filter"
    
    blocks = split_into_blocks(lines, NO_PFI_MARKER, PFI_MARKER)
    
    expected = {}
    for variant in ["no_pfi", "pfi"]:
        block = blocks[variant]
        text = "".join(block)
        expected[variant] = {}
        
        # Training and test counts
        m = re.search(r"Training points:\s+(\d+)", text)
        if m:
            expected[variant]["n_train"] = safe_int(m.group(1))
        
        m = re.search(r"Test points:\s+(\d+)", text)
        if m:
            expected[variant]["n_test"] = safe_int(m.group(1))
        
        # Descriptor count
        m = re.search(r"Number of descriptors\s*=\s*(\d+)", text)
        if m:
            expected[variant]["n_descriptors"] = safe_int(m.group(1))
        
        # Regression metrics (R2, RMSE)
        m = re.search(r"(\d+)x\s+(\d+)-fold CV\s*:\s+R2\s*=\s*([\d.eE+\-]+)", text)
        if m:
            expected[variant]["r2_cv"] = safe_float(m.group(3))
        
        m = re.search(r"Test\s*:\s+R2\s*=\s*([\d.eE+\-]+)", text)
        if m:
            expected[variant]["r2_test"] = safe_float(m.group(1))
        
        m = re.search(r"(\d+)x\s+(\d+)-fold CV\s*:.*?RMSE\s*=\s*([\d.eE+\-]+)", text)
        if m:
            expected[variant]["rmse_cv"] = safe_float(m.group(3))
        
        m = re.search(r"Test\s*:.*?RMSE\s*=\s*([\d.eE+\-]+)", text)
        if m:
            expected[variant]["rmse_test"] = safe_float(m.group(1))
        
        # Classification metrics (MCC)
        m = re.search(r"(\d+)x\s+(\d+)-fold CV\s*:.*?MCC\s*=\s*([\d.eE+\-]+)", text)
        if m:
            expected[variant]["mcc_cv"] = safe_float(m.group(3))
        
        m = re.search(r"Test\s*:.*?MCC\s*=\s*([\d.eE+\-]+)", text)
        if m:
            expected[variant]["mcc_test"] = safe_float(m.group(1))
    
    return expected

print("✓ extract_predict_parity() defined")

## Cell 6: Extract VERIFY Evidence from Source Files

Parse VERIFY_data.dat to extract test verdicts and sorted CV arrays.

In [ ]:
def extract_verify_parity(verify_dat_path: Path) -> dict:
    """Extract VERIFY test results for parity comparison.
    
    Returns dict with keys: no_pfi, pfi
    Each contains: test_results, passed_tests, unclear_tests, failed_tests,
                   flawed_mod_score, sorted_cv_rmse, sorted_cv_r2, sorted_cv_mcc
    """
    with open(verify_dat_path) as f:
        lines = f.readlines()
    
    NO_PFI_MARKER = "Starting model with all variables (No PFI)"
    PFI_MARKER = "Starting model with PFI filter"
    
    blocks = split_into_blocks(lines, NO_PFI_MARKER, PFI_MARKER)
    
    expected = {}
    for variant in ["no_pfi", "pfi"]:
        block = blocks[variant]
        text = "".join(block)
        expected[variant] = {}
        
        # Test verdicts (y_mean, y_shuffle, onehot)
        test_pattern = re.compile(
            r"[ox\-]\s+(y_mean|y_shuffle|onehot):\s+(PASSED|UNCLEAR|FAILED)"
        )
        matches = test_pattern.findall(text)
        
        if matches:
            results = [f"{test}:{verdict}" for test, verdict in matches]
            expected[variant]["test_results"] = results
            expected[variant]["passed_tests"] = sum(1 for _, v in matches if v == "PASSED")
            expected[variant]["unclear_tests"] = sum(1 for _, v in matches if v == "UNCLEAR")
            expected[variant]["failed_tests"] = sum(1 for _, v in matches if v == "FAILED")
            flawed = expected[variant]["unclear_tests"] * (-1) + expected[variant]["failed_tests"] * (-2)
            expected[variant]["flawed_mod_score"] = flawed
        else:
            expected[variant]["test_results"] = []
            expected[variant]["passed_tests"] = 0
            expected[variant]["unclear_tests"] = 0
            expected[variant]["failed_tests"] = 0
            expected[variant]["flawed_mod_score"] = 0
        
        # Sorted CV arrays
        m = re.search(r"Sorted\s+CV\s*:.*?RMSE\s*=\s*(\[[^\]]+\])", text, re.DOTALL)
        if m:
            try:
                expected[variant]["sorted_cv_rmse"] = json.loads(m.group(1))
            except:
                expected[variant]["sorted_cv_rmse"] = None
        
        m = re.search(r"Sorted\s+CV\s*:.*?R2\s*=\s*(\[[^\]]+\])", text, re.DOTALL)
        if m:
            try:
                expected[variant]["sorted_cv_r2"] = json.loads(m.group(1))
            except:
                expected[variant]["sorted_cv_r2"] = None
        
        m = re.search(r"Sorted\s+CV\s*:.*?MCC\s*=\s*(\[[^\]]+\])", text, re.DOTALL)
        if m:
            try:
                expected[variant]["sorted_cv_mcc"] = json.loads(m.group(1))
            except:
                expected[variant]["sorted_cv_mcc"] = None
    
    return expected

print("✓ extract_verify_parity() defined")

## Cell 7: Compare Extracted Values Against Source

Define functions to check if extracted run_context matches expected source values.

In [ ]:
def check_predict_parity(run_context: dict, expected: dict, pred_type: str) -> Tuple[List[str], List[str]]:
    """Check if PREDICT metrics in run_context match source file.
    
    Returns (gaps, failures):
        gaps: fields expected but not in run_context
        failures: fields where extracted != source
    """
    gaps = []
    failures = []
    
    for variant in ["no_pfi", "pfi"]:
        got = run_context["predict"][variant]
        exp = expected[variant]
        
        # Check critical fields
        for field in ["n_train", "n_test", "n_descriptors"]:
            if field in exp and exp[field] is not None:
                if got.get(field) != exp[field]:
                    failures.append(f"predict.{variant}.{field}: got {got.get(field)}, expected {exp[field]}")
        
        # Check pred-type-specific metrics
        if pred_type == "reg":
            for field in ["r2_cv", "r2_test", "rmse_cv", "rmse_test"]:
                if field in exp and exp[field] is not None:
                    got_val = got.get(field)
                    if got_val is None:
                        gaps.append(f"predict.{variant}.{field}")
                    elif abs(got_val - exp[field]) > 1e-5:
                        failures.append(f"predict.{variant}.{field}: got {got_val}, expected {exp[field]}")
        
        elif pred_type == "clas":
            for field in ["mcc_cv", "mcc_test"]:
                if field in exp and exp[field] is not None:
                    got_val = got.get(field)
                    if got_val is None:
                        gaps.append(f"predict.{variant}.{field}")
                    elif abs(got_val - exp[field]) > 1e-5:
                        failures.append(f"predict.{variant}.{field}: got {got_val}, expected {exp[field]}")
    
    return gaps, failures

def check_verify_parity(run_context: dict, expected: dict) -> Tuple[List[str], List[str]]:
    """Check if VERIFY verdicts and sorted CV in run_context match source file."""
    gaps = []
    failures = []
    
    for variant in ["no_pfi", "pfi"]:
        got = run_context["verify"][variant]
        exp = expected[variant]
        
        # Check verdict counts
        for field in ["passed_tests", "unclear_tests", "failed_tests", "flawed_mod_score"]:
            if field in exp and exp[field] is not None:
                if got.get(field) != exp[field]:
                    failures.append(f"verify.{variant}.{field}: got {got.get(field)}, expected {exp[field]}")
        
        # Check test results list
        if "test_results" in exp and exp["test_results"]:
            if got.get("test_results") != exp["test_results"]:
                failures.append(f"verify.{variant}.test_results: mismatch")
        
        # Check sorted CV arrays
        for field in ["sorted_cv_rmse", "sorted_cv_r2", "sorted_cv_mcc"]:
            if field in exp and exp[field] is not None:
                got_val = got.get(field)
                if got_val is None:
                    gaps.append(f"verify.{variant}.{field}")
                elif got_val != exp[field]:
                    failures.append(f"verify.{variant}.{field}: mismatch")
    
    return gaps, failures

print("✓ check_predict_parity() and check_verify_parity() defined")

## Cell 8: Main Parity Verification Function

Orchestrate the parity check: load run_context, extract expected values from source files, compare, and return structured result.

In [ ]:
def check_parity(run_context_path: Path, outputs_root: Path) -> ParityResult:
    """Check extraction parity for a single ROBERT run.
    
    Args:
        run_context_path: Path to run_context.json
        outputs_root: Path to outputs/ folder containing PREDICT/ and VERIFY/
    
    Returns:
        ParityResult with status (pass/fail/incomplete) and details
    """
    # Load run_context
    with open(run_context_path) as f:
        run_context = json.load(f)
    
    run_id = run_context.get("run_id", "unknown")
    pred_type = run_context.get("pred_type")
    
    gaps = []
    failures = []
    messages = []
    
    # Check PREDICT parity
    predict_dat = outputs_root / "PREDICT" / "PREDICT_data.dat"
    if run_context["available"]["predict"] and predict_dat.exists():
        expected_predict = extract_predict_parity(predict_dat)
        pred_gaps, pred_failures = check_predict_parity(run_context, expected_predict, pred_type)
        gaps.extend(pred_gaps)
        failures.extend(pred_failures)
        messages.append(f"PREDICT: checked (n_train={run_context['predict']['no_pfi'].get('n_train')})")
    elif run_context["available"]["predict"]:
        messages.append(f"PREDICT: marked available but file missing")
        gaps.append("predict_data.dat")
    
    # Check VERIFY parity
    verify_dat = outputs_root / "VERIFY" / "VERIFY_data.dat"
    if run_context["available"]["verify"] and verify_dat.exists():
        expected_verify = extract_verify_parity(verify_dat)
        verify_gaps, verify_failures = check_verify_parity(run_context, expected_verify)
        gaps.extend(verify_gaps)
        failures.extend(verify_failures)
        messages.append(f"VERIFY: checked (passed_tests={run_context['verify']['no_pfi'].get('passed_tests')})")
    elif run_context["available"]["verify"]:
        messages.append(f"VERIFY: marked available but file missing")
        gaps.append("verify_data.dat")
    
    # Determine overall status
    if failures:
        status = "fail"
        messages.insert(0, "❌ PARITY FAILED: extracted values do not match source files")
    elif gaps:
        # Gaps are OK if they're just missing optional fields
        critical_gaps = [g for g in gaps if "data.dat" in g or "n_" in g or "passed_tests" in g]
        if critical_gaps:
            status = "fail"
            messages.insert(0, f"❌ PARITY FAILED: critical fields missing: {critical_gaps}")
        else:
            status = "incomplete"
            messages.insert(0, f"⚠️  PARITY INCOMPLETE: {len(gaps)} optional fields missing (e.g., {gaps[0]})")
    else:
        status = "pass"
        messages.insert(0, "✓ PARITY PASSED: extraction matches source evidence")
    
    return ParityResult(
        run_id=run_id,
        pred_type=pred_type,
        parity_status=status,
        gaps=gaps,
        failures=failures,
        messages=messages,
    )

print("✓ check_parity() defined")

## Cell 9: Run Parity Check (Interactive Example)

When running interactively, specify the run folder below and execute this cell to verify its extraction.

In [ ]:
# Example: Check a fixture run
# Change this to point to your own run folder for audit

example_run = Path("agent/run_archive/20260516_150946__TOF_class")  # Classification fixture
run_context_path = example_run / "run_context.json"
outputs_root = example_run / "outputs"

if run_context_path.exists() and outputs_root.exists():
    result = check_parity(run_context_path, outputs_root)
    print(f"\n{result.messages[0]}")
    print(f"Run ID: {result.run_id}")
    print(f"Type: {result.pred_type}")
    print(f"Status: {result.parity_status}")
    if result.failures:
        print(f"\nFailures ({len(result.failures)}):")
        for failure in result.failures[:5]:
            print(f"  - {failure}")
    if result.gaps:
        print(f"\nGaps ({len(result.gaps)}):")
        for gap in result.gaps[:5]:
            print(f"  - {gap}")
    print(f"\nAdditional messages:")
    for msg in result.messages[1:]:
        print(f"  - {msg}")
else:
    print(f"Example run not found. Update example_run path to your run folder.")

## Future Extension: PDF Report Parity

In a future version, this notebook can also validate extracted evidence against the generated PDF report, while keeping DAT-file parity as the primary UI safeguard.

Design principle for future work:
- DAT-file parity remains mandatory for UI ground truth.
- PDF parity is an optional second check for human-facing report alignment.

To add PDF verification later:
1. Add a `parse_pdf_report()` function to extract candidate evidence from ROBERT_report.pdf.
2. Add a `check_pdf_parity()` function to compare parsed PDF values against `run_context.json`.
3. Keep DAT parity and PDF parity results separate in the returned result object.
4. Keep chat gating tied to DAT parity status unless governance changes.

This keeps the parity gate auditable, deterministic, and compatible with future report-level checks.

In [ ]:
def parse_pdf_report_placeholder(pdf_path: Path) -> dict:
    """Placeholder for future PDF extraction logic.

    This function intentionally does not implement parsing yet.
    It defines the expected contract so a future implementation can be
    added without changing the existing DAT-based parity workflow.

    Args:
        pdf_path: Path to ROBERT_report.pdf for a run.

    Returns:
        Dict with extracted report evidence fields (future schema).
    """
    raise NotImplementedError(
        "PDF parsing is not implemented yet. Use DAT-file parity as the source of truth."
    )

print("✓ PDF placeholder defined (not executed in production flow)")